# 🚗 02. Object Detection & Multi-Object Tracking
Thử nghiệm mô hình YOLO11s kết hợp thuật toán ByteTrack trên video giao thông thực tế.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from src.detection.yolo_detector import YOLODetector
from src.tracking.trajectory_manager import TrajectoryManager

detector = YOLODetector(weights="yolo11s.pt", conf_threshold=0.4)
tracker = TrajectoryManager(max_history=90)
print("Detector and Tracker initialized!")

## 1. Chạy tracking trên một video mẫu

In [ ]:
video_path = "demo/sample_videos/sample_traffic.mp4"
cap = cv2.VideoCapture(video_path if os.path.exists(video_path) else 0)

for f_idx in range(10):
    ret, frame = cap.read()
    if not ret:
        break
    res = detector.track(frame, persist=True)
    tracker.update(f_idx, res["track_ids"], res["bboxes"], res["classes"], res["confs"])
    print(f"Frame {f_idx}: Detected {len(res['track_ids'])} active objects. IDs: {res['track_ids']}")

cap.release()

## 2. Trực quan hóa quỹ đạo di chuyển (Trajectories)

In [ ]:
active_ids = tracker.get_active_tracks(current_frame=9, min_length=2)
print(f"Active track IDs with trajectory history: {active_ids}")
for tid in active_ids:
    traj = tracker.get_trajectory(tid)
    print(f" - Track #{tid} points: {len(traj)} centroids. Latest center: {traj[-1]}")